# LUDB Lead II ECG peak detection

This notebook contains an intermediate version of the ECG peak detector I developed and its evaluation on the Lobachevsky University Electrocardiography Database (LUDB), a publicly available dataset of annotated 12-lead ECG recordings. The higher metrics on my résumé come from a later evaluation of my final method.

- **Input.** LUDB v1.0.1 Lead II at 500 Hz. Apply the notebook's 60 Hz notch and 1–40 Hz Butterworth filters, then resample to the notebook's 128 Hz. Run detection over the full record, with no cropping or time alignment. Convert predicted indices back to the 500 Hz annotation grid using nearest-sample rounding.
- **Detector.** Keep Engzee with its original fallback, R refinement, the 250 ms RR filter, the Q/S and non-overlapping P/T search rules. The detector returns P and T peaks directly from their search windows. The adapter includes a guard for the case where R refinement removes every candidate. All peaks are detected on the filtered mV signal.
- **Scoring.** Use the intermediate evaluation's nearest one-to-one matching (38 samples, its rounded ±75 ms radius) and first/last manual annotations ±20 ms. Score all records and report a second summary with the intermediate evaluation's exclusions. Q/S are plotted only and the detector has no onset/offset outputs. **m** is signed prediction minus annotation. **sigma** averages sample SDs from records with at least two matches.

Data source: [PhysioNet LUDB v1.0.1](https://physionet.org/content/ludb/1.0.1/). Install the dependencies from the repository's `requirements.txt`.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from biosppy.signals import ecg as ECG
from IPython.display import display
from scipy.signal import butter, filtfilt, find_peaks, iirnotch, resample, savgol_filter, sosfiltfilt

LUDB_FOLDER = "lobachevsky-university-electrocardiography-database-1.0.1"
DATA_DIR = Path("work/ludb") / LUDB_FOLDER / "data"  # Point this to extracted LUDB data if available.
LUDB_ZIP = "https://physionet.org/content/ludb/get-zip/1.0.1/"
LEAD = "ii"
ANNOTATION_FS = 500
DETECTION_FS = 128
TOLERANCE_WINDOW_MS = 150
TOLERANCE_SAMPLES = round(TOLERANCE_WINDOW_MS / 2 / 1000 * ANNOTATION_FS)
MARGIN_MS = 20
MARGIN_SAMPLES = round(MARGIN_MS / 1000 * ANNOTATION_FS)


def load_headers(data_dir):
    headers = sorted(data_dir.glob("*.hea"), key=lambda path: int(path.stem))
    expected_ids = list(range(1, 201))
    if [int(path.stem) for path in headers] != expected_ids:
        raise ValueError(f"Expected LUDB records 1–200 in {data_dir}; found {len(headers)} headers.")
    return headers


## Peak detection functions

In [ ]:
def _notch_filter(signal, fs, freq=60.0, q=30.0):
        """Apply notch filter to remove powerline interference"""
        nyq = 0.5 * fs
        w0 = freq / nyq
        b, a = iirnotch(w0, q)
        return filtfilt(b, a, signal, axis=0)

def _butter_filter(signal, fs, lowcut=1, highcut=40.0):
        """Apply bandpass Butterworth filter"""
        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        sos = butter(4, [low, high], btype="band", output="sos")
        return sosfiltfilt(sos, signal, axis=0)

def _resample(signal, old_fs, new_fs):
        """Resample signal to new sampling rate"""
        n_samples = int(signal.shape[0] * new_fs / old_fs)
        return resample(signal, n_samples, axis=0)

def bandpass_filter(signal, fs, lowcut=0.5, highcut=20.0, order=3):
    """Stable bandpass for P-wave band (0.5-20 Hz)."""
    nyq = 0.5 * fs
    low = max(lowcut / nyq, 1e-6)
    high = min(highcut / nyq, 0.999999)
    sos = butter(order, [low, high], btype='band', output='sos')
    return sosfiltfilt(sos, signal, axis=0)

def adaptive_prominence(region, fs, factor=0.6):
    """
    Estimate a dynamic prominence threshold for find_peaks based on local noise.
    factor: scale factor (0.6-1.0) typical. Lower -> more sensitive.
    """
    if len(region) < 3:
        return 0.0
    # robust sigma estimate via MAD
    mad = np.median(np.abs(region - np.median(region)))
    sigma = 1.4826 * mad if mad > 0 else np.std(region) if np.std(region) > 0 else 1e-6
    # convert to mV-like amplitude
    return max(0.01, sigma * factor)

def rpeak_detection(ecg_data, sampling_rate=125., method_prefer='engzee'):
    """
    Robust R-peak detection:
    - Prefer biosppy.engzee_segmenter (robust to noise/baseline).
    - Fallback to christov or a simple Pan-Tompkins style energy detector.
    Returns indices of R-peaks (samples).
    """
    # try biosppy detectors first
    try:
        if method_prefer == 'engzee':
            rpeaks, = ECG.engzee_segmenter(signal=ecg_data, sampling_rate=sampling_rate)
        elif method_prefer == 'christov':
            rpeaks, = ECG.christov_segmenter(signal=ecg_data, sampling_rate=sampling_rate)
        else:
            rpeaks, = ECG.hamilton_segmenter(signal=ecg_data, sampling_rate=sampling_rate)
        # refine with correct_rpeaks (keeps positions consistent)
        rpeaks, = ECG.correct_rpeaks(signal=ecg_data, rpeaks=rpeaks, sampling_rate=sampling_rate, tol=0.03)
        return np.asarray(rpeaks, dtype=int)
    except Exception:
        # fallback: simple energy-based Pan-Tompkins-like method
        try:
            fs = int(round(sampling_rate))
            # bandpass narrow around QRS
            nyq = 0.5 * fs
            low = max(5.0 / nyq, 0.001)
            high = min(15.0 / nyq, 0.999)
            sos = butter(3, [low, high], btype='band', output='sos')
            filtered = sosfiltfilt(sos, ecg_data)
            # derivative -> square -> moving average integration
            diffed = np.diff(filtered, prepend=filtered[0])
            squared = diffed ** 2
            window = int(0.150 * fs)  # 150 ms integration window
            if window < 1: window = 1
            integrated = np.convolve(squared, np.ones(window)/window, mode='same')
            # dynamic threshold from median + std
            thr = np.median(integrated) + 0.5 * np.std(integrated)
            min_distance = int(0.25 * fs)  # refractory 250 ms
            peaks, props = find_peaks(integrated, height=thr, distance=min_distance, prominence=np.std(integrated)*0.3)
            # map to original by choosing within small neighborhood the true maxima (in filtered signal)
            refined_peaks = []
            for p in peaks:
                local_start = max(0, p - int(0.03*fs))
                local_end = min(len(filtered)-1, p + int(0.03*fs))
                local_idx = np.argmax(filtered[local_start:local_end+1])
                refined_peaks.append(local_start + local_idx)
            return np.unique(np.array(refined_peaks, dtype=int))
        except Exception:
            return np.array([], dtype=int)

def detect_qrs_complex_enhanced(ecg_signal, r_peaks, fs=125):
    """Detect Q and S peaks in QRS complex"""
    q_peaks = []
    s_peaks = []
    qrs_morphology = []

    for r_peak in r_peaks:
        # Q-wave detection window (60ms before R-peak)
        q_start = max(0, r_peak - int(0.06 * fs))
        q_end = r_peak
        q_region = ecg_signal[q_start:q_end]

        if len(q_region) > 0:
            baseline = np.mean(q_region[:min(3, len(q_region))])
            negative_points = q_region < (baseline - 0.02)
            if np.any(negative_points):
                q_candidates = np.where(negative_points)[0]
                q_idx = q_candidates[np.argmin(q_region[q_candidates])]
            else:
                q_idx = np.argmin(q_region)

            q_peak = q_start + q_idx
            q_peaks.append(q_peak)

        # S-wave detection window (80ms after R-peak)
        s_start = r_peak
        s_end = min(len(ecg_signal)-1, r_peak + int(0.08 * fs))
        s_region = ecg_signal[s_start:s_end]

        if len(s_region) > 0:
            baseline = ecg_signal[r_peak]
            negative_points = s_region < (baseline - 0.03)
            if np.any(negative_points):
                s_candidates = np.where(negative_points)[0]
                s_idx = s_candidates[np.argmin(s_region[s_candidates])]
            else:
                s_idx = np.argmin(s_region)

            s_peak = s_start + s_idx
            s_peaks.append(s_peak)

    return np.array(q_peaks), np.array(s_peaks), qrs_morphology

def find_t_peak_enhanced(t_region, fs, min_amplitude=0.05):
    """Detect T-wave peak and classify morphology"""
    if len(t_region) < 3:
        return {'peak_index': -1, 'morphology': 'Cannot Analyze', 'amplitude': 0}

    baseline_window = min(int(0.03 * fs), len(t_region) // 3)
    baseline = np.median(t_region[:baseline_window]) if baseline_window > 0 else np.median(t_region)

    detrended = t_region - baseline

    positive_peak_idx = np.argmax(detrended)
    negative_peak_idx = np.argmin(detrended)

    positive_amplitude = detrended[positive_peak_idx]
    negative_amplitude = -detrended[negative_peak_idx]

    adaptive_threshold = max(min_amplitude, np.std(detrended) * 1.5)

    if positive_amplitude >= adaptive_threshold and positive_amplitude >= negative_amplitude:
        return {
            'peak_index': positive_peak_idx,
            'morphology': 'Upright T-wave',
            'amplitude': positive_amplitude
        }
    elif negative_amplitude >= adaptive_threshold and negative_amplitude > positive_amplitude:
        return {
            'peak_index': negative_peak_idx,
            'morphology': 'Inverted T-wave',
            'amplitude': negative_amplitude
        }
    elif max(positive_amplitude, negative_amplitude) >= adaptive_threshold * 0.6:
        if positive_amplitude >= negative_amplitude:
            return {
                'peak_index': positive_peak_idx,
                'morphology': 'Low-amplitude T-wave',
                'amplitude': positive_amplitude
            }
        else:
            return {
                'peak_index': negative_peak_idx,
                'morphology': 'Low-amplitude Inverted T-wave',
                'amplitude': negative_amplitude
            }
    else:
        return {
            'peak_index': -1,
            'morphology': 'T-wave Absent/Flat',
            'amplitude': max(positive_amplitude, negative_amplitude)
        }

def find_p_peak_simple(p_region, fs, min_amplitude=0.012):
    """
    More robust P-wave detection inside a given window (p_region).
    - Applies light bandpass, smoothing and adaptive prominence peak-picking.
    - Returns {'peak_index': idx_or_-1, 'morphology': str, 'score': float}
    """
    out = {'peak_index': -1, 'morphology': 'P-wave Absent/Weak', 'score': 0.0}

    if len(p_region) < 7:
        out['morphology'] = 'Region Too Short'
        return out

    # 1) Bandpass filter to isolate P-band (0.5–20 Hz)
    try:
        filt = bandpass_filter(p_region, fs, lowcut=0.5, highcut=20.0, order=3)
    except Exception:
        filt = p_region.copy()

    # 2) Smooth with Savitzky–Golay filter
    try:
        win = 7 if len(filt) >= 7 else (len(filt) // 2) * 2 + 1
        smoothed = savgol_filter(filt, window_length=win, polyorder=2)
    except Exception:
        smoothed = filt

    # 3) Remove local baseline (median)
    baseline = np.median(smoothed[:max(3, len(smoothed)//6)])
    detrended = smoothed - baseline

    # 4) Adaptive prominence threshold
    prom = adaptive_prominence(detrended, fs, factor=0.8)
    prom = max(prom, min_amplitude)

    # 5) Find both positive and negative peaks
    pos_peaks, pos_props = find_peaks(
        detrended,
        distance=int(0.03 * fs),
        prominence=prom,
        width=1,
        height=prom * 0.5
    )
    neg_peaks, neg_props = find_peaks(
        -detrended,
        distance=int(0.03 * fs),
        prominence=prom,
        width=1,
        height=prom * 0.5
    )

    # 6) Choose best candidate
    best_idx = -1
    best_score = 0.0
    best_type = 'absent'

    if len(pos_peaks) > 0:
        prominences = pos_props.get('prominences', np.zeros(len(pos_peaks)))
        i = np.argmax(prominences)
        best_idx = int(pos_peaks[i])
        best_score = float(prominences[i])
        best_type = 'upright'

    if len(neg_peaks) > 0:
        neg_proms = neg_props.get('prominences', np.zeros(len(neg_peaks)))
        j = np.argmax(neg_proms)
        neg_score = float(neg_proms[j])
        if neg_score > best_score * 0.9:
            best_idx = int(neg_peaks[j])
            best_score = neg_score
            best_type = 'inverted'

    # 7) Fallback if no prominent peaks found
    if best_idx == -1:
        pos_idx = int(np.argmax(detrended))
        neg_idx = int(np.argmin(detrended))
        pos_amp = float(detrended[pos_idx])
        neg_amp = -float(detrended[neg_idx])
        if pos_amp >= max(min_amplitude, np.std(detrended) * 0.6):
            best_idx = pos_idx
            best_type = 'upright'
            best_score = pos_amp
        elif neg_amp >= max(min_amplitude, np.std(detrended) * 0.6):
            best_idx = neg_idx
            best_type = 'inverted'
            best_score = neg_amp

    # 8) Assign morphology label
    if best_idx >= 0:
        morphology = 'Upright P-wave' if best_type == 'upright' else 'Inverted P-wave'
        if best_score < min_amplitude * 1.2:
            morphology = 'Low-amplitude P-wave' if best_type == 'upright' else 'Low-amplitude Inverted P-wave'
        out['peak_index'] = best_idx
        out['morphology'] = morphology
        out['score'] = best_score
    else:
        out['morphology'] = 'P-wave Absent/Weak'
        out['score'] = 0.0

    return out

def detect_t_p_peaks_non_overlapping_simple(ecg_signal, r_peaks, s_peaks, fs=125):
    """Detect T and P peaks without overlap"""
    t_peaks = []
    p_peaks = []
    t_morphology = []
    p_morphology = []
    if len(r_peaks) == 0:
        return np.array([]), np.array([]), [], []

    n = len(r_peaks)
    t_end_positions = []

    # T detection (mostly reuse existing)
    for i in range(n):
        r_peak = r_peaks[i]
        s_peak = s_peaks[i] if i < len(s_peaks) else r_peak + int(0.04 * fs)

        t_start = s_peak + int(0.06 * fs)
        if i < n - 1:
            next_r = r_peaks[i + 1]
            max_t_end = next_r - int(0.18 * fs)
        else:
            max_t_end = len(ecg_signal) - 1
        t_end = min(max_t_end, s_peak + int(0.32 * fs))

        if t_start < len(ecg_signal) and t_end > t_start + 4:
            t_region = ecg_signal[t_start:t_end]
            t_result = find_t_peak_enhanced(t_region, fs)
            if t_result['peak_index'] >= 0:
                actual_t = t_start + t_result['peak_index']
                t_peaks.append(actual_t)
                t_morphology.append(t_result['morphology'])
                t_end_positions.append(actual_t + int(0.08 * fs))
            else:
                t_end_positions.append(t_end)
        else:
            t_end_positions.append(min(t_end, len(ecg_signal)-1))

    # P detection with avoidance of T overlap
    for i in range(n):
        r_peak = r_peaks[i]
        prev_t_end = t_end_positions[i-1] if i > 0 else -1

        # p window: from (prev_t_end + margin) to (r_peak - 50ms)
        p_start = max(0, int(prev_t_end + int(0.03 * fs))) if prev_t_end >= 0 else max(0, r_peak - int(0.35*fs))
        p_end = max(p_start + 6, r_peak - int(0.05 * fs))

        # reduce window if overlaps next R too closely
        if p_end > len(ecg_signal):
            p_end = len(ecg_signal) - 1
        if p_start >= p_end:
            continue

        p_region = ecg_signal[p_start:p_end]
        p_result = find_p_peak_simple(p_region, fs, min_amplitude=0.012)

        if p_result['peak_index'] >= 0:
            global_idx = p_start + int(p_result['peak_index'])
            # ensure it's not too close to an R or T peak
            if any(abs(global_idx - rr) < int(0.04 * fs) for rr in r_peaks):
                # too close to R; ignore
                continue
            # extra check: ensure not falling inside T window of previous beat
            if i > 0 and prev_t_end >= 0 and global_idx <= prev_t_end + int(0.02*fs):
                # inside previous T tail -> ignore
                continue

            p_peaks.append(global_idx)
            p_morphology.append(p_result['morphology'])

    return np.array(t_peaks), np.array(p_peaks), t_morphology, p_morphology

def refine_rpeaks(ecg_signal, rpeaks, fs, min_rr_ms=250, max_rr_ms=2000):
    """
    Post-process rpeaks:
    - remove too-close peaks (keep the one with larger local amplitude/slope)
    - remove peaks with low amplitude relative to local median
    - attempt to insert missing peaks when large gaps exist
    """
    if len(rpeaks) == 0:
        return rpeaks

    fs = int(round(fs))
    # 1) remove too-close peaks
    min_dist = int(min_rr_ms/1000.0 * fs)
    keep = []
    i = 0
    while i < len(rpeaks):
        j = i + 1
        group = [rpeaks[i]]
        while j < len(rpeaks) and rpeaks[j] - rpeaks[j-1] < min_dist:
            group.append(rpeaks[j])
            j += 1
        if len(group) == 1:
            keep.append(group[0])
        else:
            # choose best by amplitude*max_slope
            best = group[0]
            best_score = -1
            for g in group:
                # local amplitude
                amp = abs(ecg_signal[g])
                # local slope (max absolute derivative in +/-10ms)
                w = int(max(1, fs*0.01))
                l = max(0, g-w); r = min(len(ecg_signal)-1, g+w)
                slope = np.max(np.abs(np.diff(ecg_signal[l:r+1])))
                score = amp * (slope + 1e-6)
                if score > best_score:
                    best_score = score; best = g
            keep.append(best)
        i = j
    keep = np.array(sorted(keep), dtype=int)

    # 2) remove peaks with very low amplitude relative to local baseline
    amplitudes = np.abs(ecg_signal[keep])
    local_med = np.median(np.abs(ecg_signal)) + 1e-6
    mask = amplitudes >= (0.2 * local_med)  # keep peaks at least 20% of median abs amplitude
    keep = keep[mask]

    # 3) fill large gaps: if gap > 1.6 * median_rr, search for missed peak in that interval
    if len(keep) >= 2:
        rr = np.diff(keep) / fs
        median_rr = np.median(rr)
        new_peaks = []
        for idx, gap in enumerate(rr):
            if gap > 1.6 * median_rr:
                # search in interval for strong local maxima in filtered signal
                start = keep[idx] + int(0.3*fs)
                end = keep[idx+1] - int(0.3*fs)
                if end > start:
                    sub = ecg_signal[start:end]
                    if len(sub) > 3:
                        cand_rel = np.argmax(np.abs(sub))
                        cand = start + cand_rel
                        # require cand to be reasonably large and separated
                        if abs(ecg_signal[cand]) > 0.3 * np.median(np.abs(ecg_signal)):
                            new_peaks.append(cand)
        if new_peaks:
            keep = np.unique(np.sort(np.concatenate([keep, np.array(new_peaks, dtype=int)])))

    # final: ensure extremes within physiological bounds
    # remove peaks that create RR < 0.15s or > max_rr_ms
    if len(keep) == 0:
        return keep
    final = [keep[0]]
    for p in keep[1:]:
        rr_ms = (p - final[-1]) / fs * 1000
        if rr_ms < 150:
            # keep the one with larger amplitude between p and final[-1]
            cand1 = final[-1]; cand2 = p
            if abs(ecg_signal[cand2]) > abs(ecg_signal[cand1]):
                final[-1] = cand2
            # else keep existing final[-1]
        elif rr_ms > max_rr_ms:
            # too long, but we keep it (could indicate asystole or missing)
            final.append(p)
        else:
            final.append(p)
    return np.array(final, dtype=int)

def filter_short_rr_intervals(r_peaks, fs, min_rr_ms=300):
    """Filter R-peaks with short RR intervals"""
    if len(r_peaks) < 2:
        return r_peaks

    rr_intervals = np.diff(r_peaks) / fs * 1000
    mask = np.ones(len(r_peaks), dtype=bool)

    for i, rr in enumerate(rr_intervals, start=1):
        if rr < min_rr_ms:
            mask[i] = False

    return r_peaks[mask]

def filter_peaks(peaks, valid_rpeaks, fs):
    """Filter peaks to match valid R-peaks"""
    return np.array([p for p in peaks if any(abs(p - r) < int(fs*0.15) for r in valid_rpeaks)])


In [ ]:
def detect_ludb_peaks(raw_ecg):
    filtered = _butter_filter(_notch_filter(raw_ecg, ANNOTATION_FS), ANNOTATION_FS)
    signal = _resample(filtered, ANNOTATION_FS, DETECTION_FS)
    fs = DETECTION_FS

    r = rpeak_detection(signal, sampling_rate=fs, method_prefer="engzee")
    r = refine_rpeaks(signal, r, fs, min_rr_ms=250)
    r = filter_short_rr_intervals(r, fs, min_rr_ms=250)
    q, s, _ = detect_qrs_complex_enhanced(signal, r, fs)
    t, p, _, _ = detect_t_p_peaks_non_overlapping_simple(signal, r, s, fs)

    def on_annotation_grid(peaks):
        return {
            key: np.rint(np.asarray(indices) * ANNOTATION_FS / fs).astype(int)
            for key, indices in peaks.items()
        }

    predicted = on_annotation_grid({
        "P": p, "Q": filter_peaks(q, r, fs), "R": r,
        "S": filter_peaks(s, r, fs), "T": t,
    })
    return filtered, predicted


## LUDB annotation matching

In [ ]:
EVALUATION_KEYS = [
    "P_on",
    "P",
    "P_off",
    "R_on",
    "R",
    "R_off",
    "T_on",
    "T",
    "T_off",
]

DISPLAY_NAMES = {
    "P_on": "P-onset",
    "P": "P-wave",
    "P_off": "P-offset",
    "Q": "Q-wave",
    "R_on": "QRS-onset",
    "R": "R-peak",
    "R_off": "QRS-offset",
    "S": "S-wave",
    "T_on": "T-onset",
    "T": "T-wave",
    "T_off": "T-offset",
}

PLOT_ORDER_MANUAL = EVALUATION_KEYS
PLOT_ORDER_PREDICTED = [
    "P_on",
    "P",
    "P_off",
    "Q",
    "R_on",
    "R",
    "R_off",
    "S",
    "T_on",
    "T",
    "T_off",
]

PLOT_STYLES = {
    "P_on": ("tab:green", "^"),
    "P": ("tab:green", "x"),
    "P_off": ("tab:green", "v"),
    "Q": ("tab:red", "D"),
    "R_on": ("tab:orange", "^"),
    "R": ("tab:orange", "x"),
    "R_off": ("tab:orange", "v"),
    "S": ("tab:brown", "D"),
    "T_on": ("tab:purple", "^"),
    "T": ("tab:purple", "x"),
    "T_off": ("tab:purple", "v"),
}

def read_ludb_annotations(record_path, lead="ii"):
    """Read manual P, QRS/R and T fiducial annotations for one LUDB lead."""
    annotation = wfdb.rdann(str(record_path), extension=lead)
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample, dtype=int)

    manual = {}
    wave_symbols = {"P": "p", "R": "N", "T": "t"}

    for wave_name, peak_symbol in wave_symbols.items():
        peak_indices = np.flatnonzero(symbols == peak_symbol)
        peaks = []
        onsets = []
        offsets = []

        for index in peak_indices:
            peaks.append(samples[index])

            if index > 0 and symbols[index - 1] == "(":
                onsets.append(samples[index - 1])

            if index + 1 < len(symbols) and symbols[index + 1] == ")":
                offsets.append(samples[index + 1])

        manual[wave_name] = np.asarray(peaks, dtype=int)
        manual[f"{wave_name}_on"] = np.asarray(onsets, dtype=int)
        manual[f"{wave_name}_off"] = np.asarray(offsets, dtype=int)

    return manual

def match_predictions(predicted, annotated, tolerance_samples):
    """One-to-one nearest matching within the specified tolerance."""
    predicted = np.asarray(predicted, dtype=float)
    annotated = np.asarray(annotated, dtype=float)

    predicted = np.sort(predicted[np.isfinite(predicted)].astype(int))
    annotated = np.sort(annotated[np.isfinite(annotated)].astype(int))

    candidates = []
    for pred_index, pred_position in enumerate(predicted):
        for ann_index, ann_position in enumerate(annotated):
            distance = abs(pred_position - ann_position)
            if distance <= tolerance_samples:
                candidates.append((distance, pred_index, ann_index))

    candidates.sort(key=lambda item: item[0])
    matched_predictions = set()
    matched_annotations = set()
    errors = []

    for _, pred_index, ann_index in candidates:
        if pred_index in matched_predictions or ann_index in matched_annotations:
            continue

        matched_predictions.add(pred_index)
        matched_annotations.add(ann_index)
        errors.append(predicted[pred_index] - annotated[ann_index])

    tp = len(errors)
    fp = len(predicted) - tp
    fn = len(annotated) - tp

    return tp, fp, fn, np.asarray(errors, dtype=float)

def keep_points_in_window(waves, start_sample, end_sample):
    """Return wave positions that lie inside the inclusive evaluation window."""
    filtered = {}
    for key, positions in waves.items():
        positions = np.asarray(positions, dtype=float)
        positions = positions[np.isfinite(positions)].astype(int)
        filtered[key] = positions[
            (positions >= start_sample) & (positions <= end_sample)
        ]
    return filtered

def annotation_window(manual, signal_length, margin_samples):
    """Build [first manual - margin, last manual + margin] for one record."""
    all_manual_points = []
    for key in EVALUATION_KEYS:
        positions = np.asarray(manual.get(key, []), dtype=float)
        positions = positions[np.isfinite(positions)].astype(int)
        all_manual_points.extend(positions.tolist())

    if not all_manual_points:
        raise ValueError("No manual Lead II annotations were found")

    first_manual = min(all_manual_points)
    last_manual = max(all_manual_points)
    start_sample = max(0, first_manual - margin_samples)
    end_sample = min(signal_length - 1, last_manual + margin_samples)
    return start_sample, end_sample, first_manual, last_manual

def point_kind(key):
    """Map peak keys to their lowercase names. Boundary metrics are unavailable."""
    if key in ("P", "Q", "R", "S", "T"):
        return key.lower()
    if key.endswith("_on") or key.endswith("_off"):
        return None
    raise ValueError(f"Unrecognized evaluation key: {key!r}")

In [ ]:
P_EXCLUSIONS = {
    101, 103, 104, 109, 110, 111, 112, 129, 173, 8, 34, 38,
    44, 45, 51, 52, 74, 83, 88, 90, 93, 95, 96,
}


def score_record(header):
    record_id = int(header.stem)
    record_path = str(header.with_suffix(""))
    record = wfdb.rdrecord(record_path, channel_names=[LEAD])
    raw_ecg = record.p_signal[:, 0]
    if record.fs != ANNOTATION_FS or not np.all(np.isfinite(raw_ecg)):
        raise ValueError(f"Record {record_id}: expected a finite {ANNOTATION_FS} Hz signal.")

    cleaned, predicted = detect_ludb_peaks(raw_ecg)
    manual = read_ludb_annotations(record_path, lead=LEAD)
    start, end, first_manual, last_manual = annotation_window(manual, len(raw_ecg), MARGIN_SAMPLES)
    predicted = keep_points_in_window(predicted, start, end)
    manual = keep_points_in_window(manual, start, end)

    rows = []
    record_counts = {key: {"TP": 0, "FP": 0, "FN": 0} for key in EVALUATION_KEYS}
    for key in ("P", "R", "T"):
        tp, fp, fn, errors = match_predictions(predicted[key], manual[key], TOLERANCE_SAMPLES)
        record_counts[key] = {"TP": tp, "FP": fp, "FN": fn}
        rows.append({
            "record_id": record_id, "lead": LEAD,
            "window_start_sample": start, "window_end_sample": end,
            "fiducial_point": DISPLAY_NAMES[key], **record_counts[key],
            "in_reference_cohort": record_id != 35 and not (key == "P" and record_id in P_EXCLUSIONS),
            "errors_ms": errors * 1000 / ANNOTATION_FS,
        })

    window = {
        "record_id": record_id, "lead": LEAD,
        "first_manual_sample": first_manual, "last_manual_sample": last_manual,
        "window_start_sample": start, "window_end_sample": end,
        "window_start_seconds": start / ANNOTATION_FS,
        "window_end_seconds": end / ANNOTATION_FS,
    }
    figure_data = (record_id, ANNOTATION_FS, cleaned, predicted, manual, record_counts,
                   TOLERANCE_WINDOW_MS, MARGIN_MS, start, end, first_manual, last_manual)
    return rows, window, figure_data


def evaluate(data_dir):
    rows, windows, figure_data = [], [], {}
    for header in load_headers(data_dir):
        record_rows, window, figure_args = score_record(header)
        rows.extend(record_rows)
        windows.append(window)
        figure_data[window["record_id"]] = figure_args
    return pd.DataFrame(rows), pd.DataFrame(windows), figure_data


def summarize(counts):
    rows = []
    metric_columns = ["TP", "FP", "FN", "Se (%)", "PPV (%)", "F1 (%)", "m (ms)", "sigma (ms)"]
    for key in EVALUATION_KEYS:
        label = DISPLAY_NAMES[key]
        group = counts[counts["fiducial_point"] == label]
        row = {"Fiducial point": label}
        if group.empty:
            row.update(dict.fromkeys(metric_columns, np.nan))
        else:
            tp, fp, fn = map(int, group[["TP", "FP", "FN"]].sum())
            errors = np.concatenate(group["errors_ms"].tolist())
            record_sd = [np.std(e, ddof=1) for e in group["errors_ms"] if len(e) > 1]
            row.update({
                "TP": tp, "FP": fp, "FN": fn,
                "Se (%)": 100 * tp / (tp + fn) if tp + fn else np.nan,
                "PPV (%)": 100 * tp / (tp + fp) if tp + fp else np.nan,
                "F1 (%)": 200 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
                "m (ms)": np.mean(errors) if len(errors) else np.nan,
                "sigma (ms)": np.mean(record_sd) if record_sd else np.nan,
            })
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary[["TP", "FP", "FN"]] = summary[["TP", "FP", "FN"]].astype("Int64")
    return summary


In [ ]:
if DATA_DIR.is_dir():
    counts, windows, figures = evaluate(DATA_DIR)
else:
    with TemporaryDirectory() as temp_dir:
        temp = Path(temp_dir)
        archive = temp / "ludb.zip"
        urlretrieve(LUDB_ZIP, archive)
        with ZipFile(archive) as zip_file:
            zip_file.extractall(temp)
        counts, windows, figures = evaluate(temp / LUDB_FOLDER / "data")

summary_all = summarize(counts)
summary_reference = summarize(counts[counts["in_reference_cohort"]])
print(f"Processed {len(windows)}/200 records. Matching radius: {TOLERANCE_SAMPLES} samples.")
print("All 200 records")
display(summary_all.round(2).astype(object).fillna("N/A"))
print("Legacy comparison cohort (176 P records, 199 R and T records)")
display(summary_reference.round(2).astype(object).fillna("N/A"))


## Manual versus detected peaks

In [ ]:
def plot_annotations(ax, time, ecg, waves, keys, panel_title):
    """Plot one ECG panel and its manual or predicted annotations."""
    ax.plot(time, ecg, color="black", linewidth=0.8, label="ECG")

    for key in keys:
        if key not in waves:
            continue

        positions = np.asarray(waves[key], dtype=float)
        positions = positions[np.isfinite(positions)].astype(int)
        positions = positions[(positions >= 0) & (positions < len(ecg))]

        if len(positions) == 0:
            continue

        color, marker = PLOT_STYLES[key]
        ax.scatter(
            time[positions],
            ecg[positions],
            color=color,
            marker=marker,
            s=34,
            linewidths=1.6,
            label=DISPLAY_NAMES[key],
            zorder=3,
        )

    ax.set_title(panel_title, fontsize=12, loc="left")
    ax.set_ylabel("Amplitude [mV]")
    ax.grid(alpha=0.25)
    ax.legend(
        loc="upper right",
        ncol=5,
        fontsize=7,
        framealpha=0.9,
    )

def comparison_figure(
    record_id,
    fs,
    cleaned_ecg,
    predicted,
    manual,
    record_counts,
    tolerance_window_ms,
    annotation_margin_ms,
    start_sample,
    end_sample,
    first_manual,
    last_manual,
):
    """Save manual/machine comparison restricted to the annotation window."""
    time = np.arange(len(cleaned_ecg)) / fs

    fig = plt.figure(figsize=(18, 9), constrained_layout=True)
    grid = fig.add_gridspec(2, 2, width_ratios=[4.8, 1.35])
    ax_manual = fig.add_subplot(grid[0, 0])
    ax_predicted = fig.add_subplot(grid[1, 0], sharex=ax_manual)
    ax_table = fig.add_subplot(grid[:, 1])

    plot_annotations(
        ax_manual,
        time,
        cleaned_ecg,
        manual,
        PLOT_ORDER_MANUAL,
        "Manual annotations",
    )
    plot_annotations(
        ax_predicted,
        time,
        cleaned_ecg,
        predicted,
        PLOT_ORDER_PREDICTED,
        "Machine annotations (Notebook detector)",
    )

    start_seconds = start_sample / fs
    end_seconds = end_sample / fs
    first_manual_seconds = first_manual / fs
    last_manual_seconds = last_manual / fs

    ax_predicted.set_xlabel("Time [s]")
    ax_predicted.set_xlim(start_seconds, end_seconds)

    for axis in (ax_manual, ax_predicted):
        axis.axvline(
            first_manual_seconds,
            color="steelblue",
            linestyle="--",
            linewidth=0.8,
            alpha=0.65,
        )
        axis.axvline(
            last_manual_seconds,
            color="steelblue",
            linestyle="--",
            linewidth=0.8,
            alpha=0.65,
        )

    total_tp = sum(record_counts[key]["TP"] for key in EVALUATION_KEYS)
    total_fp = sum(record_counts[key]["FP"] for key in EVALUATION_KEYS)
    total_fn = sum(record_counts[key]["FN"] for key in EVALUATION_KEYS)

    table_rows = []
    row_labels = []
    for key in EVALUATION_KEYS:
        counts = record_counts[key]
        row_labels.append(DISPLAY_NAMES[key])
        table_rows.append(
            [counts["TP"], counts["FP"], counts["FN"]]
            if point_kind(key) is not None else ["N/A", "N/A", "N/A"]
        )

    row_labels.append("TOTAL")
    table_rows.append([total_tp, total_fp, total_fn])

    ax_table.axis("off")
    ax_table.set_title(
        f"Annotation-window matching\nTP={total_tp}   FP={total_fp}   FN={total_fn}",
        fontsize=13,
        pad=20,
    )
    table = ax_table.table(
        cellText=table_rows,
        rowLabels=row_labels,
        colLabels=["TP", "FP", "FN"],
        cellLoc="center",
        rowLoc="left",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.05, 1.55)

    total_row_index = len(table_rows)
    for column_index in range(-1, 3):
        cell = table.get_celld().get((total_row_index, column_index))
        if cell is not None:
            cell.set_facecolor("#e5e7eb")
            cell.set_text_props(weight="bold")

    fig.suptitle(
        f"LUDB Record {record_id} — Lead II — "
        f"{tolerance_window_ms} ms total matching window "
        f"(±{tolerance_window_ms / 2:g} ms)",
        fontsize=17,
        fontweight="bold",
    )
    fig.text(
        0.5,
        0.005,
        f"Evaluation interval: {start_seconds:.3f}–{end_seconds:.3f} s "
        f"(manual span ±{annotation_margin_ms:g} ms). "
        "Machine detections outside this interval are excluded. "
        "Blue dashed lines mark the first and last manual annotations. "
        "Q/S are display only. Boundary metrics are N/A: the notebook detector returns peaks.",
        ha="center",
        fontsize=8.5,
        color="#444444",
    )

    return fig

In [ ]:
from io import BytesIO
from IPython.display import Image
example_record_id = 1  # Change this to inspect any record from 1 to 200.
fig = comparison_figure(*figures[example_record_id])
with BytesIO() as buffer:
    fig.savefig(buffer, format="png", dpi=110, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
plt.close(fig)


## Per-record counts and evaluation windows

In [ ]:
display(counts.drop(columns="errors_ms").head(9))
display(windows.head(3))
